# 🔍 Teste Direto: dbutils e Extração de Contexto

## Objetivo Simples: 
Testar **diretamente** a extração de URL e token do Databricks usando dbutils, sem complexidade adicional.

### 🎯 Testes:
1. **Verificar se dbutils está disponível**
2. **Extrair workspace URL via dbutils**  
3. **Extrair token de admin via dbutils**
4. **Validar credenciais extraídas**

## 1. 🧪 Teste Básico: dbutils Disponível?

In [ ]:
# Teste 1: Verificar dbutils
print("🧪 Teste 1: Verificação de dbutils")
print("=" * 35)

# Verificar se dbutils está disponível
print("1️⃣ Verificando dbutils em globals...")
if 'dbutils' in globals():
    print("✅ dbutils encontrado em globals()")
    dbutils_available = True
    dbutils_obj = globals()['dbutils']
    print(f"   Tipo: {type(dbutils_obj)}")
else:
    print("❌ dbutils NÃO encontrado em globals()")
    dbutils_available = False

# Verificar se podemos acessar dbutils diretamente
print("\n2️⃣ Testando acesso direto a dbutils...")
try:
    # Tentar usar dbutils diretamente (deve estar disponível em notebooks Databricks)
    test_result = dbutils.fs.ls("/")
    print("✅ dbutils.fs.ls('/') funcionou - dbutils está funcional")
    print(f"   Resultado: {len(test_result)} itens encontrados")
    dbutils_functional = True
except Exception as e:
    print(f"❌ Erro ao usar dbutils: {e}")
    dbutils_functional = False

# Status final
print(f"\n📊 Status dbutils:")
print(f"   Disponível: {'✅' if dbutils_available else '❌'}")
print(f"   Funcional: {'✅' if dbutils_functional else '❌'}")

if dbutils_available and dbutils_functional:
    print(f"🎉 dbutils está PRONTO para extrair contexto!")
else:
    print(f"❌ dbutils tem problemas - investigar ambiente")

## 2. 🌐 Extrair Workspace URL via dbutils

In [ ]:
# Teste 2: Extrair workspace URL
print("🌐 Teste 2: Extração de Workspace URL")
print("=" * 40)

try:
    # Método direto conforme sua sugestão
    print("1️⃣ Extraindo URL via dbutils.notebook.entry_point...")
    databricks_instance = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
    
    if databricks_instance:
        print(f"✅ Workspace URL extraída: {databricks_instance}")
        
        # Garantir formato HTTPS
        if not databricks_instance.startswith('https://'):
            full_url = f"https://{databricks_instance}"
            print(f"🔗 URL completa: {full_url}")
        else:
            full_url = databricks_instance
            print(f"🔗 URL já completa: {full_url}")
        
        # Salvar para próximos testes
        globals()['extracted_workspace_url'] = full_url
        
        # Validar formato da URL
        if '.cloud.databricks.com' in full_url or '.azuredatabricks.net' in full_url:
            print("✅ Formato de URL válido para Databricks")
        else:
            print("⚠️ Formato de URL não reconhecido")
            
    else:
        print("❌ Workspace URL está vazia ou None")
        
except Exception as e:
    print(f"❌ Erro ao extrair workspace URL: {e}")
    print(f"   Tipo do erro: {type(e).__name__}")
    import traceback
    traceback.print_exc()

# Teste método alternativo
print(f"\n2️⃣ Teste método alternativo (Spark context)...")
try:
    if 'spark' in globals():
        spark_conf = spark.sparkContext.getConf()
        workspace_from_spark = spark_conf.get('spark.databricks.workspaceUrl')
        
        if workspace_from_spark:
            print(f"✅ URL via Spark: {workspace_from_spark}")
        else:
            print("❌ spark.databricks.workspaceUrl não encontrada")
    else:
        print("❌ Spark não disponível")
        
except Exception as e:
    print(f"❌ Erro no método Spark: {e}")

# Resultado final
if 'extracted_workspace_url' in globals():
    print(f"\n🎯 SUCESSO! Workspace URL: {globals()['extracted_workspace_url']}")
else:
    print(f"\n❌ FALHA na extração de workspace URL")

## 3. 🔑 Extrair Token de Admin via dbutils

In [ ]:
# Teste 3: Extrair token de admin
print("🔑 Teste 3: Extração de Token de Admin")
print("=" * 40)

try:
    # Método direto conforme sua sugestão
    print("1️⃣ Extraindo token via dbutils.notebook.entry_point...")
    admin_token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
    
    if admin_token:
        # Mascarar token para segurança
        masked_token = f"{admin_token[:15]}...{admin_token[-8:]}" if len(admin_token) > 23 else "***"
        print(f"✅ Token extraído: {masked_token}")
        print(f"   Comprimento: {len(admin_token)} caracteres")
        
        # Verificar se parece com token Databricks
        if admin_token.startswith('dapi'):
            print("✅ Formato típico de token Databricks (dapi...)")
        else:
            print("⚠️ Formato de token não reconhecido")
        
        # Salvar para próximos testes
        globals()['extracted_admin_token'] = admin_token
        
        # Criar headers conforme sua sugestão
        headers = {"Authorization": f"Bearer {admin_token}"}
        globals()['auth_headers'] = headers
        print("✅ Headers de autorização criados")
        
    else:
        print("❌ Token está vazio ou None")
        
except Exception as e:
    print(f"❌ Erro ao extrair token: {e}")
    print(f"   Tipo do erro: {type(e).__name__}")
    import traceback
    traceback.print_exc()

# Resultado final
if 'extracted_admin_token' in globals():
    print(f"\n🎯 SUCESSO! Token extraído e headers criados")
    print(f"   Token: {globals()['extracted_admin_token'][:15]}...")
    print(f"   Headers: Authorization configurado")
else:
    print(f"\n❌ FALHA na extração de token")

## 4. ✅ Validar Credenciais Extraídas

In [ ]:
# Teste 4: Validar credenciais extraídas
print("✅ Teste 4: Validação de Credenciais")
print("=" * 38)

# Verificar o que conseguimos extrair
credentials_status = {
    'workspace_url': 'extracted_workspace_url' in globals(),
    'admin_token': 'extracted_admin_token' in globals(),
    'auth_headers': 'auth_headers' in globals()
}

print("📊 Status das Credenciais:")
for cred, status in credentials_status.items():
    status_icon = "✅" if status else "❌"
    print(f"   {status_icon} {cred}: {'Extraído' if status else 'Não extraído'}")

# Se temos ambos, fazer teste de conectividade simples
if credentials_status['workspace_url'] and credentials_status['admin_token']:
    print(f"\n🧪 Teste de Conectividade Básica:")
    
    try:
        import requests
        
        workspace_url = globals()['extracted_workspace_url']
        headers = globals()['auth_headers']
        
        # Teste simples: obter informações do usuário atual
        current_user_url = f"{workspace_url}/api/2.0/preview/scim/v2/Me"
        
        print(f"🔗 Testando: {current_user_url}")
        
        response = requests.get(current_user_url, headers=headers, timeout=10)
        
        if response.status_code == 200:
            user_info = response.json()
            print(f"✅ Conectividade OK!")
            print(f"   Usuário: {user_info.get('userName', 'N/A')}")
            print(f"   Nome: {user_info.get('displayName', 'N/A')}")
            
        else:
            print(f"⚠️ Resposta inesperada: {response.status_code}")
            print(f"   Mensagem: {response.text[:100]}...")
            
    except ImportError:
        print("⚠️ Biblioteca 'requests' não disponível - pulando teste de conectividade")
    except Exception as e:
        print(f"❌ Erro no teste de conectividade: {e}")

# Resultado final
successful_extractions = sum(credentials_status.values())
total_extractions = len(credentials_status)

print(f"\n📈 Score de Extração: {successful_extractions}/{total_extractions}")

if successful_extractions == total_extractions:
    print(f"🎉 PERFEITO! Todas as credenciais foram extraídas")
    print(f"🚀 Pronto para integrar com Dino SDK")
elif successful_extractions >= 2:
    print(f"🎯 PARCIAL! Credenciais principais extraídas")
    print(f"⚡ Suficiente para prosseguir")
else:
    print(f"❌ FALHA! Credenciais insuficientes")
    print(f"🔧 Investigar problemas no ambiente")

# Exibir credenciais finais (mascaradas)
if 'extracted_workspace_url' in globals():
    print(f"\n🌐 Workspace URL: {globals()['extracted_workspace_url']}")
if 'extracted_admin_token' in globals():
    token = globals()['extracted_admin_token']
    masked = f"{token[:15]}...{token[-8:]}"
    print(f"🔑 Admin Token: {masked}")

## 5. 🔧 Integração com Dino SDK (Se Credenciais OK)

In [ ]:
# Teste 5: Integração com Dino SDK usando credenciais extraídas
print("🔧 Teste 5: Integração com Dino SDK")
print("=" * 35)

# Verificar se temos credenciais
if 'extracted_workspace_url' in globals() and 'extracted_admin_token' in globals():
    print("✅ Credenciais disponíveis - configurando Dino SDK")
    
    # Configurar variáveis de ambiente temporárias
    import os
    
    workspace_url = globals()['extracted_workspace_url']
    admin_token = globals()['extracted_admin_token']
    
    # Definir env vars temporárias para o Dino SDK
    os.environ['DATABRICKS_HOST'] = workspace_url
    os.environ['DATABRICKS_TOKEN'] = admin_token
    
    print(f"🔗 DATABRICKS_HOST configurado: {workspace_url}")
    print(f"🔑 DATABRICKS_TOKEN configurado: {admin_token[:15]}...")
    
    # Tentar usar Dino SDK agora
    try:
        print(f"\n🦕 Testando Dino SDK com credenciais extraídas...")
        
        from src.keyvault_config import KeyVaultConfigManager
        
        # Criar manager com credenciais extraídas
        kv_manager = KeyVaultConfigManager(
            keyvault_name="dino-shared-keyvault",
            catalog_name="dino_catalog",
            schema_name="teste_dbutils_direto"
        )
        
        print(f"✅ KeyVaultConfigManager criado")
        
        # Tentar inicializar cliente
        kv_manager._initialize_databricks_client()
        
        if hasattr(kv_manager, 'databricks_client') and kv_manager.databricks_client:
            print(f"✅ Cliente Databricks inicializado!")
            
            # Testar operação básica
            current_user = kv_manager.databricks_client.current_user.me()
            print(f"👤 Usuário: {current_user.user_name}")
            
            print(f"\n🎉 SUCESSO TOTAL!")
            print(f"   ✅ dbutils funcionou")
            print(f"   ✅ Credenciais extraídas")
            print(f"   ✅ Dino SDK inicializado")
            print(f"   ✅ API funcionando")
            
        else:
            print(f"❌ Cliente Databricks não foi inicializado")
            
    except ImportError:
        print(f"⚠️ Dino SDK não instalado - mas credenciais foram extraídas!")
        print(f"💡 Instale: %pip install dino_sdk-1.1.0-py3-none-any.whl")
        
    except Exception as e:
        print(f"❌ Erro no Dino SDK: {e}")
        print(f"💡 Mas as credenciais foram extraídas corretamente!")
        
else:
    print(f"❌ Credenciais não disponíveis")
    print(f"💡 Execute os testes anteriores primeiro")

# Limpeza (opcional)
print(f"\n🧹 Limpeza de env vars temporárias...")
if 'DATABRICKS_HOST' in os.environ and 'extracted_workspace_url' in globals():
    # Manter apenas se foi definido por nós
    print(f"   Mantendo DATABRICKS_HOST para uso posterior")
if 'DATABRICKS_TOKEN' in os.environ and 'extracted_admin_token' in globals():
    print(f"   Mantendo DATABRICKS_TOKEN para uso posterior")

## 📋 Resumo Final

Este notebook testa **diretamente** os métodos básicos de extração de contexto:

### ✅ **Testes Realizados:**
1. **dbutils disponibilidade** - Verificar se dbutils está acessível
2. **Workspace URL** - `dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()`
3. **Admin Token** - `dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()`
4. **Validação** - Testar conectividade básica
5. **Integração** - Usar credenciais com Dino SDK

### 🎯 **Objetivo:**
Provar que o método básico de extração funciona, isolando o problema do Dino SDK.

### 🔧 **Próximos Passos:**
Se os testes passarem, o problema está na implementação do Dino SDK, não no dbutils.